# E-comers 360 | Evolución del negocio

## Pregunta de negocio

¿Cómo evolucionan las órdenes, el valor vendido, el flete y la satisfacción en el tiempo?

In [22]:
paquetes <- c(
  "readr", "dplyr", "tidyr", "stringr",
  "lubridate", "ggplot2", "purrr", "scales"
)

por_instalar <- paquetes[
  !vapply(paquetes, requireNamespace, logical(1), quietly = TRUE)
]

if (length(por_instalar) > 0) {
  install.packages(por_instalar, repos = "https://cloud.r-project.org")
}

invisible(lapply(paquetes, library, character.only = TRUE))

options(scipen = 999)

In [23]:
ruta_datos <- "../data/olist"

orders <- read_csv(
  file.path(ruta_datos, "olist_orders_dataset.csv"),
  show_col_types = FALSE
)

items <- read_csv(
  file.path(ruta_datos, "olist_order_items_dataset.csv"),
  show_col_types = FALSE
)

reviews <- read_csv(
  file.path(ruta_datos, "olist_order_reviews_dataset.csv"),
  show_col_types = FALSE
)

In [24]:
# 2.1 Dimensiones y estructura

tibble(
  tabla = c("orders", "items", "reviews"),
  filas = c(nrow(orders), nrow(items), nrow(reviews)),
  columnas = c(ncol(orders), ncol(items), ncol(reviews))
)

glimpse(orders)
glimpse(items)
glimpse(reviews)

tabla,filas,columnas
<chr>,<int>,<int>
orders,99441,8
items,112650,7
reviews,99224,7


Rows: 99,441
Columns: 8
$ order_id                      <chr> "e481f51cbdc54678b7cc49136f2d6af7", "53c…
$ customer_id                   <chr> "9ef432eb6251297304e76186b10a928d", "b08…
$ order_status                  <chr> "delivered", "delivered", "delivered", "…
$ order_purchase_timestamp      <dttm> 2017-10-02 10:56:33, 2018-07-24 20:41:3…
$ order_approved_at             <dttm> 2017-10-02 11:07:15, 2018-07-26 03:24:2…
$ order_delivered_carrier_date  <dttm> 2017-10-04 19:55:00, 2018-07-26 14:31:0…
$ order_delivered_customer_date <dttm> 2017-10-10 21:25:13, 2018-08-07 15:27:4…
$ order_estimated_delivery_date <dttm> 2017-10-18, 2018-08-13, 2018-09-04, 201…
Rows: 112,650
Columns: 7
$ order_id            <chr> "00010242fe8c5a6d1ba2dd792cb16214", "00018f77f2f03…
$ order_item_id       <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1,…
$ product_id          <chr> "4244733e06e7ecb4970a6e2683c13e61", "e5f2d52b80218…
$ seller_id           <chr> "48436dade18ac8b2bce089ec2a041202", "dd7ddc

In [25]:
# 2.2 Nulos y granularidad

tibble(
  variable = c(
    "order_purchase_timestamp",
    "price",
    "freight_value",
    "review_score"
  ),
  nulos = c(
    sum(is.na(orders$order_purchase_timestamp)),
    sum(is.na(items$price)),
    sum(is.na(items$freight_value)),
    sum(is.na(reviews$review_score))
  )
)

tibble(
  tabla = c("orders", "items", "reviews"),
  filas = c(nrow(orders), nrow(items), nrow(reviews)),
  ordenes_unicas = c(
    n_distinct(orders$order_id),
    n_distinct(items$order_id),
    n_distinct(reviews$order_id)
  )
)

variable,nulos
<chr>,<int>
order_purchase_timestamp,0
price,0
freight_value,0
review_score,0


tabla,filas,ordenes_unicas
<chr>,<int>,<int>
orders,99441,99441
items,112650,98666
reviews,99224,98673


In [26]:
# 3.1 Resumen de ítems por orden

items_orden <- items %>%
  group_by(order_id) %>%
  summarise(
    valor_vendido = sum(price),
    flete = sum(freight_value),
    n_items = n(),
    .groups = "drop"
  )

In [27]:
# 3.2 Resumen de reseñas por orden

reviews_orden <- reviews %>%
  group_by(order_id) %>%
  summarise(
    satisfaccion = mean(review_score),
    n_reviews = n(),
    .groups = "drop"
  )

In [28]:
# 4.1 Vista 360 a nivel de orden

vista_orden <- orders %>%
  left_join(items_orden, by = "order_id") %>%
  left_join(reviews_orden, by = "order_id")

In [29]:
# 4.2 Validación de la vista

vista_orden %>%
  summarise(
    filas = n(),
    ordenes_unicas = n_distinct(order_id),
    sin_items = sum(is.na(valor_vendido)),
    sin_review = sum(is.na(satisfaccion))
  )

filas,ordenes_unicas,sin_items,sin_review
<int>,<int>,<int>,<int>
99441,99441,775,768


In [30]:
# Revisar órdenes sin información asociada

vista_orden %>%
  filter(is.na(valor_vendido)) %>%
  count(order_status, sort = TRUE)

vista_orden %>%
  filter(is.na(satisfaccion)) %>%
  count(order_status, sort = TRUE)

order_status,n
<chr>,<int>
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


order_status,n
<chr>,<int>
delivered,646
shipped,75
canceled,20
unavailable,14
processing,6
invoiced,5
created,2


In [31]:
vista_orden %>%
  group_by(order_status) %>%
  summarise(
    ordenes = n(),
    valor_vendido = sum(valor_vendido, na.rm = TRUE),
    flete = sum(flete, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(ordenes))

order_status,ordenes,valor_vendido,flete
<chr>,<int>,<dbl>,<dbl>
delivered,96478,13221498.11,2198275.64
shipped,1107,150727.44,26401.90
canceled,625,95235.27,10650.45
unavailable,609,2007.69,132.80
invoiced,314,61526.37,7462.38
processing,301,60439.22,8954.89
created,5,0.00,0.00
approved,2,209.60,31.48


In [32]:
orders %>%
  summarise(
    fecha_min = min(order_purchase_timestamp),
    fecha_max = max(order_purchase_timestamp)
  )

fecha_min,fecha_max
<dttm>,<dttm>
2016-09-04 21:15:19,2018-10-17 17:30:18


In [33]:
vista_orden %>%
  filter(order_status == "delivered") %>%
  mutate(mes = floor_date(order_purchase_timestamp, "month")) %>%
  count(mes, name = "ordenes") %>%
  arrange(mes)

mes,ordenes
<dttm>,<int>
2016-09-01,1
2016-10-01,265
2016-12-01,1
2017-01-01,750
2017-02-01,1653
2017-03-01,2546
2017-04-01,2303
2017-05-01,3546
2017-06-01,3135
